In [1]:
import pandas as pd
import numpy as np
import sys
import os
import joblib

# import some libraries for training
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import torch.nn as nn
import torch.optim as optim

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)


In [2]:
df = pd.read_csv(os.path.join('datasets', 'flextrack_phase1_train.csv'))
display(df.head())

,Site,Timestamp_Local,Dry_Bulb_Temperature_C,Global_Horizontal_Radiation_W/m2,Building_Power_kW,Demand_Response_Flag,Demand_Response_Capacity_kW
0,siteA,2019-01-01 00:00:00,22.20,0.0,4.8,0,0.0
1,siteA,2019-01-01 00:15:00,22.27,0.0,4.8,0,0.0
2,siteA,2019-01-01 00:30:00,22.35,0.0,4.8,0,0.0
3,siteA,2019-01-01 00:45:00,22.42,0.0,4.8,0,0.0
4,siteA,2019-01-01 01:00:00,22.50,0.0,4.8,0,0.0


In [3]:
df.groupby("Demand_Response_Flag")['Site'].count()

Demand_Response_Flag
-1      2262
 0    102011
 1       847
Name: Site, dtype: int64

In [4]:
def preprocess_data(df_in):
    df = df_in.copy()
    # Convert 'Timestamp' to datetime
    df['Timestamp'] = pd.to_datetime(df['Timestamp_Local'])
    # Extract datetime features
    df['Hour'] = df['Timestamp'].dt.hour
    df['Day'] = df['Timestamp'].dt.day
    df['Month'] = df['Timestamp'].dt.month
    df['Weekday'] = df['Timestamp'].dt.weekday
    df['Minute'] = df['Timestamp'].dt.minute
    # Build seasonal features
    df['Is_Weekend'] = df['Weekday'].isin([5, 6]).astype(int)
    df['Is_Summer'] = df['Month'].isin([6, 7, 8]).astype(int)
    df['Is_Winter'] = df['Month'].isin([12, 1, 2]).astype(int)
    df['Is_Spring'] = df['Month'].isin([3, 4, 5]).astype(int)
    df['Is_Fall'] = df['Month'].isin([9, 10, 11]).astype(int)
    # Create hour of day categories
    df['Morning'] = ((df['Hour'] >= 5) & (df['Hour'] < 11)).astype(int)
    df['Afternoon'] = ((df['Hour'] >= 11) & (df['Hour'] < 18)).astype(int)
    df['Evening'] = ((df['Hour'] >= 18) & (df['Hour'] < 24)).astype(int)
    df['Nighttime'] = ((df['Hour'] >= 0) & (df['Hour'] < 5)).astype(int)
    # drop unused columns
    df.drop(columns=['Timestamp_Local','Timestamp','Site','Demand_Response_Capacity_kW'], inplace=True)
    # Fix target variable (instead of -1 make it 2)
    df['Demand_Response_Flag'] = df['Demand_Response_Flag'].replace(-1, 2)
    return df

df = preprocess_data(df)

In [5]:
display(df.head(), df.tail())

,Dry_Bulb_Temperature_C,Global_Horizontal_Radiation_W/m2,Building_Power_kW,Demand_Response_Flag,Hour,Day,Month,Weekday,Minute,Is_Weekend,Is_Summer,Is_Winter,Is_Spring,Is_Fall,Morning,Afternoon,Evening,Nighttime
0,22.20,0.0,4.8,0,0,1,1,1,0,0,0,1,0,0,0,0,0,1
1,22.27,0.0,4.8,0,0,1,1,1,15,0,0,1,0,0,0,0,0,1
2,22.35,0.0,4.8,0,0,1,1,1,30,0,0,1,0,0,0,0,0,1
3,22.42,0.0,4.8,0,0,1,1,1,45,0,0,1,0,0,0,0,0,1
4,22.50,0.0,4.8,0,1,1,1,1,0,0,0,1,0,0,0,0,0,1


,Dry_Bulb_Temperature_C,Global_Horizontal_Radiation_W/m2,Building_Power_kW,Demand_Response_Flag,Hour,Day,Month,Weekday,Minute,Is_Weekend,Is_Summer,Is_Winter,Is_Spring,Is_Fall,Morning,Afternoon,Evening,Nighttime
105115,20.57,0.0,56.33,0,22,31,12,6,45,1,0,1,0,0,0,0,1,0
105116,20.60,0.0,56.33,0,23,31,12,6,0,1,0,1,0,0,0,0,1,0
105117,20.75,0.0,56.33,0,23,31,12,6,15,1,0,1,0,0,0,0,1,0
105118,20.90,0.0,56.33,0,23,31,12,6,30,1,0,1,0,0,0,0,1,0
105119,21.05,0.0,56.33,0,23,31,12,6,45,1,0,1,0,0,0,0,1,0


In [6]:
# Assume the target column is 'Demand_Response_Flag' (3 classes: 0, 1, 2)
# If not, replace with the correct target column

# Prepare features and target
X = df.drop(columns=['Demand_Response_Flag']).values
y = df['Demand_Response_Flag'].values

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# Convert to torch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Define neural network
class Net(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(input_dim, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, 16)
        self.fc3 = nn.Linear(16, num_classes)
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

input_dim = X_train.shape[1]
num_classes = 3

# create a neural net model
model = Net(input_dim, num_classes)

# Calculate class weights to handle class imbalance
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
# Update criterion to use class weights
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# configure optimizer
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training loop
epochs = 500
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

# Evaluate
model.eval()
with torch.no_grad():
    outputs = model(X_test_tensor)
    _, predicted = torch.max(outputs, 1)
    accuracy = (predicted == y_test_tensor).float().mean().item()
    print(f"Test Accuracy: {accuracy:.4f}")

Epoch 10/500, Loss: 0.7903
Epoch 20/500, Loss: 0.6023
Epoch 30/500, Loss: 0.5503
Epoch 40/500, Loss: 0.5031
Epoch 50/500, Loss: 0.4527
Epoch 60/500, Loss: 0.4104
Epoch 70/500, Loss: 0.3742
Epoch 80/500, Loss: 0.3489
Epoch 90/500, Loss: 0.3307
Epoch 100/500, Loss: 0.3162
Epoch 110/500, Loss: 0.3035
Epoch 120/500, Loss: 0.2916
Epoch 130/500, Loss: 0.2802
Epoch 140/500, Loss: 0.2689
Epoch 150/500, Loss: 0.2587
Epoch 160/500, Loss: 0.2494
Epoch 170/500, Loss: 0.2391
Epoch 180/500, Loss: 0.2304
Epoch 190/500, Loss: 0.2220
Epoch 200/500, Loss: 0.2186
Epoch 210/500, Loss: 0.2076
Epoch 220/500, Loss: 0.2016
Epoch 230/500, Loss: 0.1954
Epoch 240/500, Loss: 0.1903
Epoch 250/500, Loss: 0.1850
Epoch 260/500, Loss: 0.1814
Epoch 270/500, Loss: 0.1765
Epoch 280/500, Loss: 0.1747
Epoch 290/500, Loss: 0.1706
Epoch 300/500, Loss: 0.1668
Epoch 310/500, Loss: 0.1637
Epoch 320/500, Loss: 0.1597
Epoch 330/500, Loss: 0.1606
Epoch 340/500, Loss: 0.1572
Epoch 350/500, Loss: 0.1526
Epoch 360/500, Loss: 0.1503
E

In [7]:
joblib.dump(scaler, 'scaler.pkl')
# Save the trained model to a file
torch.save(model.state_dict(), 'trained_model.pth')

In [8]:
# Load the trained model from file
model_loaded = Net(input_dim, num_classes)
model_loaded.load_state_dict(torch.load('trained_model.pth'))
model_loaded.eval()

Net(
  (fc1): Linear(in_features=17, out_features=32, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=32, out_features=16, bias=True)
  (fc3): Linear(in_features=16, out_features=3, bias=True)
)

In [9]:
df.head()

,Dry_Bulb_Temperature_C,Global_Horizontal_Radiation_W/m2,Building_Power_kW,Demand_Response_Flag,Hour,Day,Month,Weekday,Minute,Is_Weekend,Is_Summer,Is_Winter,Is_Spring,Is_Fall,Morning,Afternoon,Evening,Nighttime
0,22.20,0.0,4.8,0,0,1,1,1,0,0,0,1,0,0,0,0,0,1
1,22.27,0.0,4.8,0,0,1,1,1,15,0,0,1,0,0,0,0,0,1
2,22.35,0.0,4.8,0,0,1,1,1,30,0,0,1,0,0,0,0,0,1
3,22.42,0.0,4.8,0,0,1,1,1,45,0,0,1,0,0,0,0,0,1
4,22.50,0.0,4.8,0,1,1,1,1,0,0,0,1,0,0,0,0,0,1


In [10]:
# Calculate the distribution of predicted classes
unique_pred, counts_pred = torch.unique(predicted, return_counts=True)
distribution_pred = dict(zip(unique_pred.tolist(), counts_pred.tolist()))
print("Distribution in predicted:", distribution_pred)

Distribution in predicted: {0: 18741, 1: 652, 2: 1631}
